In [9]:
!uv pip install -q feast[aws]==0.40.1 s3fs==2023.12.2 pandas==2.3.3

In [10]:
!uv pip freeze | grep -E "feast|s3fs|pandas"

feast==0.40.1
pandas==2.3.3
s3fs==2023.12.2


In [13]:
from datetime import datetime

import feast
import pandas as pd

fs = feast.FeatureStore(repo_path=".")

# Entity dataframe — tells Feast WHICH entities to fetch and AT WHAT TIME.
# This is the point-in-time correct join key.
# In production: this comes from your loan application events table.
# Here: we reconstruct it from the gold parquet customer_ids.
entity_df = pd.DataFrame(
    {
        "customer_id": range(149390),  # all entities
        "event_timestamp": pd.Timestamp("2026-03-14 00:00:00", tz="UTC"),
    }
)

# Fetch historical features — reads S3 offline store directly, no DynamoDB needed.
training_df = fs.get_historical_features(
    entity_df=entity_df,
    features=fs.get_feature_service("credit_risk_v1"),
).to_df()

print(training_df.shape)
print(training_df.head())

(149390, 19)
   customer_id           event_timestamp  \
0        25820 2026-03-14 00:00:00+00:00   
1        25839 2026-03-14 00:00:00+00:00   
2        25856 2026-03-14 00:00:00+00:00   
3        25859 2026-03-14 00:00:00+00:00   
4        25873 2026-03-14 00:00:00+00:00   

   revolving_utilization_of_unsecured_lines  age  \
0                                       NaN  NaN   
1                                       NaN  NaN   
2                                       NaN  NaN   
3                                       NaN  NaN   
4                                       NaN  NaN   

   number_of_time30_59_days_past_due_not_worse  debt_ratio  monthly_income  \
0                                          NaN         NaN             NaN   
1                                          NaN         NaN             NaN   
2                                          NaN         NaN             NaN   
3                                          NaN         NaN             NaN   
4                  